<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/10.Checkpointing_and_Recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 10 — Checkpointing and Recovery



Bayangkan training yang sudah berjalan enam jam di Colab, lalu runtime terputus. Tanpa checkpoint, enam jam itu hilang. Dengan checkpoint yang setengah jadi, misalnya bobot tersimpan tetapi keadaan optimizer tidak, training memang bisa jalan lagi, tetapi lintasannya sudah berbeda dari training yang tidak pernah terputus.

Bab ini membahas cara menyimpan dan memulihkan training: apa saja yang perlu disimpan, seberapa sering, dalam format apa, dan bagaimana bangkit setelah proses mati. Karena topiknya kegagalan, notebook ini mengujinya dengan kegagalan sungguhan. Proses training dijalankan sebagai subprocess, dihentikan dengan `SIGTERM` dan `SIGKILL`, lalu dilanjutkan dari checkpoint. Kurva loss hasil resume kemudian dibandingkan dengan run yang tidak pernah diganggu.

Urutannya mengikuti bab: trainer dasar, strategi frekuensi, format file, recovery dari kegagalan, checkpoint terdistribusi, versioning, dan otomasi. Listing buku yang tidak bisa dijalankan apa adanya ditaruh di kotak kutipan pendek, diikuti versi yang bekerja.

In [1]:
import collections
import copy
import importlib.util
import io
import json
import math
import os
import pickle
import re
import shutil
import signal
import socket
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

ALLOW_CPU = os.environ.get("CH10_ALLOW_CPU", "0") == "1"   # True hanya untuk uji logika di CPU
WORK = Path(os.environ.get("CH10_WORKDIR", "ch10_workspace")).resolve()
WORK.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(WORK / "hf_home"))    # cache unduhan ikut di folder kerja bab

for module, package in [("transformers", "transformers"), ("datasets", "datasets"), ("safetensors", "safetensors")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import torch
import transformers
from datasets import load_dataset
from safetensors.torch import save_model as st_save_model
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

transformers.logging.set_verbosity_error()
print(f"python {sys.version.split()[0]} | torch {torch.__version__} (CUDA build {torch.version.cuda}) | transformers {transformers.__version__}")
print("folder kerja:", WORK)

python 3.13.16 | torch 2.11.0+cu130 (CUDA build 13.0) | transformers 5.20.2
folder kerja: /content/ch10_workspace


Sel berikut adalah gerbang GPU. Ia tidak cukup bertanya `torch.cuda.is_available()`, tetapi benar-benar mengalokasikan matriks di GPU, mengalikannya, menunggu GPU selesai (`synchronize`), dan memeriksa hasilnya bernilai hingga. Dari *compute capability* kartu, sel ini juga memilih presisi untuk trainer tangguh nanti: T4 (7.5) tidak punya bf16 native, jadi memakai fp16 dengan `GradScaler`; kartu Ampere (8.0) ke atas memakai bf16.

In [2]:
def cuda_gate():
    if not torch.cuda.is_available():
        return None
    device = torch.device("cuda")
    a = torch.randn(512, 512, device=device)
    b = a @ a.T
    torch.cuda.synchronize()
    if not torch.isfinite(b).all():
        raise RuntimeError("Matmul uji di GPU menghasilkan nilai non-finite. Ganti runtime lalu jalankan ulang.")
    del a, b
    return device


DEVICE = cuda_gate()
if DEVICE is not None:
    major, minor = torch.cuda.get_device_capability(DEVICE)
    free, total = torch.cuda.mem_get_info()
    CHILD_AMP = "bf16" if major >= 8 else "fp16"
    PROFILE = "gpu"
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)} | compute capability {major}.{minor} | "
          f"memori bebas {free / 2**30:.1f} dari {total / 2**30:.1f} GiB")
elif ALLOW_CPU:
    DEVICE = torch.device("cpu")
    CHILD_AMP = "off"
    PROFILE = "cpu-reduced"
    print("PERINGATAN: CUDA tidak tersedia. Berjalan di CPU karena ALLOW_CPU=True; ini uji logika, bukan hasil GPU.")
else:
    raise RuntimeError(
        "CUDA tidak tersedia atau tidak berfungsi. Di Colab: Runtime > Change runtime type > pilih GPU (mis. T4), "
        "lalu jalankan ulang dari atas. Untuk uji logika di CPU, set ALLOW_CPU = True (profil dikurangi).")

CFG = {
    "gpu": {"child_steps": 60, "kill_after": 35, "save_every": 10, "throttle_s": 0.05},
    "cpu-reduced": {"child_steps": 40, "kill_after": 25, "save_every": 10, "throttle_s": 0.02},
}[PROFILE]
torch.manual_seed(42)
print(f"device: {DEVICE} | profil: {PROFILE} | AMP trainer tangguh: {CHILD_AMP} | {CFG}")

GPU: Tesla T4 | compute capability 7.5 | memori bebas 14.4 dari 14.6 GiB
device: cuda | profil: gpu | AMP trainer tangguh: fp16 | {'child_steps': 60, 'kill_after': 35, 'save_every': 10, 'throttle_s': 0.05}


## Data latihan mini

Untuk membuktikan resume benar, training harus bisa diulang persis: inisialisasi sama, urutan batch sama. Kita ambil 4.000 baris pertama WikiText-2, memecahnya menjadi token dengan tokenizer GPT-2, lalu memetakan 4.095 token paling sering ke vocab ringkas berukuran 4.096. Token sisanya digabung menjadi satu ID "lainnya". Vocab kecil membuat model mini cepat dilatih. Hasilnya dipotong menjadi blok 64 token; 480 blok pertama dipakai untuk training dan juga disimpan ke disk, karena proses anak yang nanti kita hentikan perlu membaca data yang sama.

Model di bagian-bagian awal dilatih dalam fp32 di GPU. Presisi campuran tidak dibutuhkan untuk model sekecil ini, dan fp32 membuat perbandingan keadaan model sebelum dan sesudah load mudah dibaca. AMP baru dipakai di trainer tangguh, karena di sana state `GradScaler` termasuk hal yang harus ikut tersimpan.

In [3]:
SEQ_LEN, VOCAB, N_TRAIN_BLOCKS = 64, 4096, 480

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
texts = [t for t in dataset["text"][:4000] if t.strip()]
raw_ids = []
for token_ids in tokenizer(texts)["input_ids"]:
    raw_ids.extend(token_ids + [tokenizer.eos_token_id])

freq = collections.Counter(raw_ids)
remap = {tok: i + 1 for i, (tok, _) in enumerate(freq.most_common(VOCAB - 1))}   # ID 0 = "lainnya"
ids = torch.tensor([remap.get(t, 0) for t in raw_ids])
n_blocks = len(ids) // SEQ_LEN
assert n_blocks >= N_TRAIN_BLOCKS, f"korpus terlalu pendek: {n_blocks} blok"
train_blocks = ids[: N_TRAIN_BLOCKS * SEQ_LEN].view(N_TRAIN_BLOCKS, SEQ_LEN)
torch.save({"blocks": train_blocks, "vocab": VOCAB}, WORK / "corpus_child.pt")

print(f"{len(ids):,} token | {n_blocks:,} blok tersedia, {N_TRAIN_BLOCKS} dipakai | "
      f"token di luar vocab ringkas: {(ids == 0).float().mean():.1%}")


def make_mini(dropout=0.0):
    cfg = GPT2Config(vocab_size=VOCAB, n_positions=SEQ_LEN, n_embd=64, n_layer=2, n_head=2,
                     resid_pdrop=dropout, embd_pdrop=dropout, attn_pdrop=dropout,
                     bos_token_id=0, eos_token_id=0)
    return GPT2LMHeadModel(cfg).to(DEVICE)


def batch(step, size=8):
    start = (step * size) % N_TRAIN_BLOCKS
    return train_blocks[start:start + size].to(DEVICE)


def train_step(model, optimizer, x):
    # satu step fp32: forward, backward, update; model dan input wajib di device yang sama
    assert x.device == next(model.parameters()).device, "input dan model berada di device berbeda"
    loss = model(input_ids=x, labels=x).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    value = loss.item()
    if not math.isfinite(value):
        raise RuntimeError(f"loss non-finite: {value}")
    return value

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

264,762 token | 4,136 blok tersedia, 480 dipakai | token di luar vocab ringkas: 13.9%


Angka persentase terakhir menunjukkan berapa banyak token yang jatuh ke ID "lainnya". Di validasi lokal penyusun hasilnya 13,9%, jadi sekitar 86% token yang dilihat model tetap kata asli dan masih ada pola bahasa yang bisa dipelajari meski vocab-nya dipangkas. Angka ini ditentukan oleh data dan tokenizer, bukan oleh device, sehingga seharusnya sama di Colab.

## Kenapa checkpoint penting

Checkpoint adalah foto keadaan training pada satu titik: bobot model, keadaan optimizer, dan posisi progres (epoch, step). Kelas `LLMTrainer` di buku menyimpan ketiganya ke satu file `.pt` lewat `torch.save`, lalu memuatnya kembali dengan `load_checkpoint`.

> **Listing buku: blok pemakaian yang perlu dilengkapi**
>
> ```python
> # Simulating training loop
> for epoch in range(10):
>     for step in range(1000):
>         # ... training code ...
>         if step % 100 == 0:
>             trainer.save_checkpoint(epoch, step, loss.item())
> ```
>
> Dijalankan apa adanya, blok ini berhenti dengan `NameError: name 'trainer' is not defined`: `trainer` tidak pernah dibuat dan `loss` hanya ada di dalam komentar. Kelasnya sendiri benar. Dua sel berikut mendefinisikan kelas itu, lalu membuat `trainer` dan `loss` dari training sungguhan.

In [4]:
class LLMTrainer:
    def __init__(self, model, optimizer, checkpoint_dir='checkpoints'):
        self.model = model
        self.optimizer = optimizer
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)

    def save_checkpoint(self, epoch, step, loss):
        checkpoint = {
            'epoch': epoch,
            'step': step,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': loss,
        }
        checkpoint_path = os.path.join(self.checkpoint_dir, f'checkpoint_epoch_{epoch}_step_{step}.pt')
        torch.save(checkpoint, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")
        return checkpoint_path                     # tambahan: dibutuhkan rotasi di bagian berikut

    def load_checkpoint(self, checkpoint_path):
        device = next(self.model.parameters()).device
        checkpoint = torch.load(checkpoint_path, map_location=device)   # tambahan: map_location
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return checkpoint['epoch'], checkpoint['step'], checkpoint['loss']

Uji berikut lebih ketat daripada "load tidak error". Model dilatih 31 step dengan checkpoint tiap 10 step. Keadaan pada step 30 dicatat, lalu training dilanjutkan lima step lagi supaya bobot dan momen optimizer bergeser. Setelah `load_checkpoint`, keduanya harus kembali identik dengan catatan step 30.

In [5]:
torch.manual_seed(42)
model = make_mini()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
trainer = LLMTrainer(model, optimizer, checkpoint_dir=str(WORK / 'ck_basic'))
print(f"model mini: {model.num_parameters():,} parameter di {next(model.parameters()).device}")

model.train()
for step in range(31):
    loss = train_step(model, optimizer, batch(step))
    if step % 10 == 0:
        trainer.save_checkpoint(epoch=0, step=step, loss=loss)
weights_30 = {k: v.detach().clone() for k, v in model.state_dict().items()}
moment_30 = optimizer.state_dict()['state'][0]['exp_avg'].clone()

for name in sorted(os.listdir(trainer.checkpoint_dir), key=lambda s: int(re.findall(r'\d+', s)[-1])):
    print(f"  {name}: {os.path.getsize(os.path.join(trainer.checkpoint_dir, name)) / 1e6:.2f} MB")

for step in range(31, 36):                          # lima step tambahan: keadaan bergeser
    train_step(model, optimizer, batch(step))
moved = not torch.equal(model.transformer.h[0].attn.c_attn.weight, weights_30['transformer.h.0.attn.c_attn.weight'])

epoch, step, loss_ck = trainer.load_checkpoint(os.path.join(trainer.checkpoint_dir, 'checkpoint_epoch_0_step_30.pt'))
weights_ok = all(torch.equal(v, weights_30[k]) for k, v in model.state_dict().items())
moment_ok = torch.equal(optimizer.state_dict()['state'][0]['exp_avg'], moment_30)
print(f"dimuat: epoch {epoch}, step {step}, loss {loss_ck:.4f}")
print(f"bobot sempat bergeser: {moved} | setelah load, bobot identik: {weights_ok} | momen AdamW identik: {moment_ok}")
assert moved and weights_ok and moment_ok

model mini: 366,336 parameter di cuda:0
Checkpoint saved: /content/ch10_workspace/ck_basic/checkpoint_epoch_0_step_0.pt
Checkpoint saved: /content/ch10_workspace/ck_basic/checkpoint_epoch_0_step_10.pt
Checkpoint saved: /content/ch10_workspace/ck_basic/checkpoint_epoch_0_step_20.pt
Checkpoint saved: /content/ch10_workspace/ck_basic/checkpoint_epoch_0_step_30.pt
  checkpoint_epoch_0_step_0.pt: 4.43 MB
  checkpoint_epoch_0_step_10.pt: 4.43 MB
  checkpoint_epoch_0_step_20.pt: 4.43 MB
  checkpoint_epoch_0_step_30.pt: 4.43 MB
dimuat: epoch 0, step 30, loss 6.8669
bobot sempat bergeser: True | setelah load, bobot identik: True | momen AdamW identik: True


Model mini punya 366.336 parameter. Dalam fp32 (4 byte per angka) bobotnya sekitar 1,47 MB, tetapi setiap file checkpoint sekitar 4,43 MB. Selisihnya adalah state AdamW: dua tensor seukuran model, yaitu rata-rata gradien dan rata-rata kuadrat gradien. Jadi checkpoint berisi kira-kira tiga kali ukuran bobot. Ukuran ini ditentukan oleh jumlah parameter, sehingga sama di CPU maupun GPU. Bagian optimizer itu bukan pemborosan: tanpa dua tensor tersebut, AdamW yang dilanjutkan harus membangun ulang statistik gradiennya, dan langkah-langkah awal setelah resume berbeda dari training aslinya.

Ada satu perubahan sejak buku ditulis. Mulai PyTorch 2.6, `torch.load` memakai `weights_only=True` secara default. Sel berikut menyimpan checkpoint yang berisi objek Python buatan sendiri, lalu mencoba memuatnya dengan dua cara. Penolakan di percobaan pertama memang perilaku yang ingin ditunjukkan, jadi ditangkap secara spesifik dan dicetak.

In [6]:
class TrainingMeta:                                  # objek custom: config, catatan eksperimen, dll.
    def __init__(self):
        self.note = "run eksperimen #7"


legacy_path = WORK / "legacy_ckpt.pt"
torch.save({"meta": TrainingMeta(), "w": torch.randn(3)}, legacy_path)
try:
    torch.load(legacy_path)
    print("dimuat tanpa penolakan (PyTorch < 2.6: default weights_only=False)")
except pickle.UnpicklingError as e:
    print("ditolak:", type(e).__name__, "-", str(e).splitlines()[0][:80])
ck = torch.load(legacy_path, weights_only=False)     # hanya untuk file yang asal-usulnya kamu percaya
print("dengan weights_only=False:", ck["meta"].note)

ditolak: UnpicklingError - Weights only load failed. This file can still be loaded, to do so you have two o
dengan weights_only=False: run eksperimen #7


`weights_only=True` hanya mengizinkan tensor, dict, list, dan tipe dasar. Alasannya keamanan: file pickle bisa menjalankan kode sembarang saat dimuat. Checkpoint `LLMTrainer` tetap bisa dimuat dengan default baru karena isinya hanya state dict dan angka. Checkpoint yang menyimpan objek custom perlu `weights_only=False`, dan keputusan itu sebaiknya diambil secara sadar, hanya untuk file yang kamu percaya.

## Frekuensi dan strategi penyimpanan

Menyimpan di setiap step terlalu mahal, sedangkan menyimpan terlalu jarang membuat banyak progres hilang saat crash. Buku menawarkan tiga strategi di `AdvancedLLMTrainer`: **rotasi** (simpan berkala, hapus yang tertua kalau jumlahnya melebihi batas), **berbasis waktu** (simpan setiap N menit), dan **model terbaik** (simpan hanya saat loss memecahkan rekor).

> **Listing buku: rotasi yang gagal di checkpoint keenam**
>
> ```python
> def save_checkpoint(self, epoch, step, loss):
>     checkpoint_path = super().save_checkpoint(epoch, step, loss)
>     self.checkpoints.append(checkpoint_path)
>     if len(self.checkpoints) > self.max_checkpoints:
>         oldest_checkpoint = self.checkpoints.pop(0)
>         os.remove(oldest_checkpoint)
> ```
>
> `save_checkpoint` milik `LLMTrainer` di buku tidak mengembalikan path, jadi `checkpoint_path` bernilai `None`. Lima save pertama lancar; save keenam memicu rotasi dan berhenti dengan `TypeError` karena `os.remove(None)`. Bug seperti ini lolos dari uji singkat yang jarang menyimpan lebih dari lima file. `LLMTrainer` di atas sudah memakai `return checkpoint_path`, sehingga kelas berikut bekerja.

Nilai loss di demo ini angka buatan, karena yang diuji adalah logika penyimpanan, bukan training.

In [7]:
class AdvancedLLMTrainer(LLMTrainer):
    def __init__(self, model, optimizer, checkpoint_dir='checkpoints', max_checkpoints=5):
        super().__init__(model, optimizer, checkpoint_dir)
        self.max_checkpoints = max_checkpoints
        self.checkpoints = []

    def save_checkpoint(self, epoch, step, loss):
        checkpoint_path = super().save_checkpoint(epoch, step, loss)
        self.checkpoints.append(checkpoint_path)
        if len(self.checkpoints) > self.max_checkpoints:
            oldest_checkpoint = self.checkpoints.pop(0)
            os.remove(oldest_checkpoint)
            print(f"Removed old checkpoint: {oldest_checkpoint}")
        return checkpoint_path

    def save_checkpoint_by_time(self, epoch, step, loss, interval_minutes=60):
        current_time = time.time()
        if (not hasattr(self, 'last_checkpoint_time') or
                current_time - self.last_checkpoint_time >= interval_minutes * 60):
            self.save_checkpoint(epoch, step, loss)
            self.last_checkpoint_time = current_time

    def save_best_checkpoint(self, epoch, step, loss):
        if not hasattr(self, 'best_loss') or loss < self.best_loss:
            self.best_loss = loss
            torch.save({'epoch': epoch, 'step': step,
                        'model_state_dict': self.model.state_dict(),
                        'optimizer_state_dict': self.optimizer.state_dict(),
                        'loss': loss},
                       os.path.join(self.checkpoint_dir, 'best_model.pt'))
            print(f"  [best] step {step}: loss {loss:.1f}")


ck_adv = WORK / 'ck_adv'
shutil.rmtree(ck_adv, ignore_errors=True)
rotating = AdvancedLLMTrainer(model, optimizer, checkpoint_dir=str(ck_adv), max_checkpoints=5)
for step in range(0, 80, 10):                       # 8 save, batas 5 file
    rotating.save_checkpoint(epoch=0, step=step, loss=5.0 - step * 0.01)
kept = sorted((n for n in os.listdir(ck_adv) if n.startswith('checkpoint_')), key=lambda s: int(re.findall(r'\d+', s)[-1]))
print("tersisa:", kept)

print("model terbaik pada loss yang naik-turun:")
for i, value in enumerate([5.0, 4.2, 4.6, 3.9, 4.4, 3.7, 4.1]):
    rotating.save_best_checkpoint(epoch=0, step=i, loss=value)

ck_time = WORK / 'ck_time'
shutil.rmtree(ck_time, ignore_errors=True)
timed = AdvancedLLMTrainer(model, optimizer, checkpoint_dir=str(ck_time))
timed.save_checkpoint_by_time(0, 0, 4.0, interval_minutes=1 / 60)   # interval 1 detik: tersimpan
timed.save_checkpoint_by_time(0, 1, 3.9, interval_minutes=1 / 60)   # belum 1 detik: dilewati
time.sleep(1.1)
timed.save_checkpoint_by_time(0, 2, 3.8, interval_minutes=1 / 60)   # interval lewat: tersimpan
timed_files = sorted(os.listdir(ck_time))
print("simpan berbasis waktu:", timed_files)
assert len(kept) == 5 and rotating.best_loss == 3.7 and len(timed_files) == 2

Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_0.pt
Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_10.pt
Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_20.pt
Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_30.pt
Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_40.pt
Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_50.pt
Removed old checkpoint: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_0.pt
Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_60.pt
Removed old checkpoint: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_10.pt
Checkpoint saved: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_70.pt
Removed old checkpoint: /content/ch10_workspace/ck_adv/checkpoint_epoch_0_step_20.pt
tersisa: ['checkpoint_epoch_0_step_30.pt', 'checkpoint_epoch_0_step_40.pt', 'checkpoint_epoch_0_step_50.pt', 'check

Rotasi menyimpan delapan kali, tetapi folder hanya berisi lima file terakhir. Model terbaik menulis empat kali dari tujuh nilai loss, tepat saat rekor pecah (5,0 → 4,2 → 3,9 → 3,7). Gerbang waktu melewatkan panggilan kedua yang datang sebelum interval satu detik habis.

Ketiganya punya peran berbeda. Rotasi menjaga disk tetap terkendali dan selalu menyediakan checkpoint terbaru untuk recovery, tetapi bisa membuang model yang bagus saat loss naik-turun. Simpan berbasis waktu cocok untuk run berhari-hari karena biaya I/O-nya terjadwal. Model terbaik membantu memilih bobot akhir, tetapi bisa tertipu batch yang kebetulan mudah, dan tidak menggantikan checkpoint terbaru untuk melanjutkan training. Dalam praktik ketiganya sering dipakai bersamaan.

## Format checkpoint yang efisien

Untuk model besar, ukuran file menjadi masalah. Buku mengusulkan kompresi ZIP: setiap state dict diserialisasi ke memori, lalu ditulis ke arsip dengan kompresi `ZIP_DEFLATED`.

> **Listing buku: loader dengan nama metode terbelah**
>
> ```python
> self.model.load_state_dict(
>     checkpoint['model_state_dict'])
> self.optimizer.load_state_ dict(checkpoint['optimizer_state_dict'])
> ```
>
> Spasi di tengah `load_state_ dict` membuat Python berhenti dengan `SyntaxError` sebelum apa pun dijalankan. Pola yang sama muncul di blok pemakaian buku (`trainer.load_checkpoint_ efficient(...)`). Versi di bawah memakai nama yang utuh.

In [8]:
class EfficientLLMTrainer(AdvancedLLMTrainer):
    def save_checkpoint_efficient(self, epoch, step, loss, name=None):
        checkpoint = {
            'epoch': epoch,
            'step': step,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': loss,
        }
        stem = name or f'checkpoint_epoch_{epoch}_step_{step}'   # tambahan: nama opsional untuk autosave thread
        checkpoint_path = os.path.join(self.checkpoint_dir, f'{stem}.zip')
        with zipfile.ZipFile(checkpoint_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for key, value in checkpoint.items():
                if isinstance(value, dict):
                    buffer = io.BytesIO()
                    torch.save(value, buffer)
                    zipf.writestr(f'{key}.pt', buffer.getvalue())
                else:
                    zipf.writestr(f'{key}.txt', str(value))
        print(f"Efficient checkpoint saved: {checkpoint_path}")
        return checkpoint_path                      # tambahan: dibutuhkan versioning

    def load_checkpoint_efficient(self, checkpoint_path):
        device = next(self.model.parameters()).device
        checkpoint = {}
        with zipfile.ZipFile(checkpoint_path, 'r') as zipf:
            for filename in zipf.namelist():
                with zipf.open(filename) as f:
                    if filename.endswith('.pt'):
                        checkpoint[filename[:-3]] = torch.load(io.BytesIO(f.read()), map_location=device)
                    else:
                        key = filename[:-4]
                        value = f.read().decode('utf-8')
                        checkpoint[key] = int(value) if key in ['epoch', 'step'] else float(value)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])   # nama metode utuh
        return checkpoint['epoch'], checkpoint['step'], checkpoint['loss']

Format ini kita uji pada model yang lebih realistis untuk soal ukuran: GPT-2 dua lapis dengan vocab GPT-2 penuh (50.257 token), sekitar 6,8 juta parameter. Model dilatih tiga step di GPU dengan token acak, cukup untuk mengisi momen AdamW. Lalu checkpoint ZIP disimpan, keadaan digeser dua step, dan dimuat kembali. Epoch, step, dan loss yang ditulis ke file hanyalah label.

In [9]:
torch.manual_seed(7)
big = GPT2LMHeadModel(GPT2Config(vocab_size=50257, n_positions=96, n_embd=128, n_layer=2, n_head=2)).to(DEVICE)
big_opt = torch.optim.AdamW(big.parameters(), lr=1e-3)
xb = torch.randint(0, 50257, (4, 96), device=DEVICE)
for _ in range(3):
    train_step(big, big_opt, xb)

btrainer = EfficientLLMTrainer(big, big_opt, checkpoint_dir=str(WORK / 'ck_eff'))
zip_path = btrainer.save_checkpoint_efficient(epoch=5, step=500, loss=9.87)
w_ref = big.transformer.h[0].attn.c_attn.weight.detach().clone()
m_ref = big_opt.state_dict()['state'][0]['exp_avg'].clone()

for _ in range(2):                                   # geser bobot dan momen agar load benar-benar diuji
    train_step(big, big_opt, xb)
epoch, step, loss_ck = btrainer.load_checkpoint_efficient(zip_path)
same_w = torch.equal(big.transformer.h[0].attn.c_attn.weight, w_ref)
same_m = torch.equal(big_opt.state_dict()['state'][0]['exp_avg'], m_ref)
print(f"{big.num_parameters() / 1e6:.2f} juta parameter di {next(big.parameters()).device}")
print(f"dimuat: epoch {epoch}, step {step}, loss {loss_ck} | bobot identik: {same_w} | momen AdamW identik: {same_m}")
assert same_w and same_m

Efficient checkpoint saved: /content/ch10_workspace/ck_eff/checkpoint_epoch_5_step_500.zip
6.84 juta parameter di cuda:0
dimuat: epoch 5, step 500, loss 9.87 | bobot identik: True | momen AdamW identik: True


Roundtrip ZIP lengkap: bobot dan momen optimizer kembali identik. Pertanyaan berikutnya, seberapa efisien format ini?

Hitung dulu ekspektasinya. Bobot fp32 berukuran 6,84 juta × 4 byte ≈ 27,4 MB. AdamW menambah dua tensor seukuran itu, sehingga checkpoint lengkap sekitar 82 MB. Sel berikut membandingkan enam cara menyimpan model yang sama. Pekerjaannya adalah serialisasi, I/O disk, dan kompresi di CPU; GPU tidak mempercepat apa pun di sini. Waktu bergantung pada disk dan hanya diukur sekali, jadi anggap indikatif dan fokus pada ukuran.

In [10]:
fmt_dir = WORK / 'formats'
fmt_dir.mkdir(exist_ok=True)
msd = big.state_dict()
full = {'model_state_dict': msd, 'optimizer_state_dict': big_opt.state_dict()}
rows = []


def measure(label, filename, write, baseline):
    path = fmt_dir / filename
    t0 = time.perf_counter()
    write(path)
    rows.append((label, os.path.getsize(path), time.perf_counter() - t0, baseline))


def write_book_zip(path):
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as z:
        for key, value in full.items():
            buffer = io.BytesIO()
            torch.save(value, buffer)
            z.writestr(f'{key}.pt', buffer.getvalue())


naive16 = {k: v.half() for k, v in msd.items()}
measure("torch.save fp32, model+optimizer", 'raw.pt', lambda p: torch.save(full, p), 'full')
measure("ZIP_DEFLATED ala buku, model+optimizer", 'book.zip', write_book_zip, 'full')
measure("torch.save fp32, model saja", 'model.pt', lambda p: torch.save(msd, p), 'model')
measure("fp16 naif {k: v.half()}, model saja", 'fp16_naive.pt', lambda p: torch.save(naive16, p), 'model')
measure("fp16 via model.half(), model saja", 'fp16_model.pt', lambda p: torch.save(copy.deepcopy(big).half().state_dict(), p), 'model')
measure("safetensors fp32, model saja", 'model.safetensors', lambda p: st_save_model(big, str(p)), 'model')

base = {'full': rows[0][1], 'model': rows[2][1]}
print(f"{'format':<40}{'MB':>8}{'detik':>8}   relatif terhadap fp32")
for label, size, seconds, ref in rows:
    print(f"{label:<40}{size / 1e6:>8.1f}{seconds:>8.2f}   {size / base[ref]:.0%}")
print("wte dan lm_head berbagi memori di model:", big.lm_head.weight.data_ptr() == big.transformer.wte.weight.data_ptr())
print("berbagi memori setelah half() per tensor:", naive16['lm_head.weight'].data_ptr() == naive16['transformer.wte.weight'].data_ptr())

del naive16, msd, full, btrainer, big, big_opt, xb
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print(f"memori GPU teralokasi setelah dibersihkan: {torch.cuda.memory_allocated() / 2**20:.1f} MiB")

format                                        MB   detik   relatif terhadap fp32
torch.save fp32, model+optimizer            82.1    0.13   100%
ZIP_DEFLATED ala buku, model+optimizer      75.6    3.92   92%
torch.save fp32, model saja                 27.4    0.03   100%
fp16 naif {k: v.half()}, model saja         26.6    0.04   97%
fp16 via model.half(), model saja           13.7    0.04   50%
safetensors fp32, model saja                27.4    0.04   100%
wte dan lm_head berbagi memori di model: True
berbagi memori setelah half() per tensor: False
memori GPU teralokasi setelah dibersihkan: 50.0 MiB


Tabel itu memuat tiga pelajaran.

**Kompresi ZIP hampir tidak mengecilkan bobot fp32.** Di validasi lokal penyusun (CPU), checkpoint lengkap turun dari 82,1 MB menjadi 75,5 MB, hanya 8% lebih kecil, sementara waktunya 18 kali lebih lama (run historis notebook asal: 82,1 → 75,6 MB). Penyebabnya, bit-bit angka float hasil training nyaris acak, sehingga kompresi lossless tidak punya banyak pola untuk diperas.

**`half()` per tensor bisa gagal mengecilkan file.** GPT-2 memakai *weight tying*: matriks embedding input (`wte`) dan proyeksi output (`lm_head`) adalah tensor yang sama. `torch.save` cukup menyimpannya sekali. Dict comprehension `{k: v.half()}` membuat salinan terpisah untuk setiap nama, jadi matriks 6,4 juta parameter itu tersimpan dua kali dan file fp16 hanya sedikit lebih kecil dari fp32 (97% dari ukuran fp32 di validasi lokal). `model.half()` mengubah model secara utuh, tying tetap terjaga, dan ukurannya mendarat di sekitar 50%.

**safetensors tidak mengompres.** Ukurannya setara `torch.save` untuk presisi yang sama. Kelebihannya ada di keamanan (tanpa pickle) dan pemuatan yang cepat lewat memory mapping; itu sebabnya format ini menjadi default ekosistem Hugging Face. Jadi ukuran checkpoint dikendalikan oleh presisi dan deduplikasi, bukan oleh ZIP.

## Pulih dari kegagalan

`RobustLLMTrainer` di buku menambahkan tiga kemampuan: menangkap sinyal `SIGINT`/`SIGTERM` lalu menyimpan checkpoint sebelum keluar, menyimpan otomatis secara berkala, dan melanjutkan dari checkpoint terbaru saat dijalankan ulang. Dua istilah perlu dibedakan dulu.

- `SIGTERM` adalah permintaan berhenti yang sopan. Proses boleh menangkapnya, merapikan pekerjaan, lalu keluar. Preemption di layanan cloud biasanya diawali sinyal ini dengan masa tenggang.
- `SIGKILL` menghentikan proses seketika. Sinyal ini tidak bisa ditangkap, jadi yang tersisa hanya checkpoint yang sudah tertulis sebelumnya.

Skrip anak di bawah memuat mekanisme itu dengan beberapa perbaikan dan tambahan:

1. Checkpoint berisi **seluruh** keadaan yang memengaruhi step berikutnya: bobot, optimizer, scheduler learning rate, `GradScaler` (faktor skala loss untuk fp16), state RNG CPU dan GPU (dropout bergantung padanya), serta kursor data. Di sini urutan batch adalah fungsi dari nomor step, jadi kursor data cukup berupa nomor step. Kalau memakai `DataLoader` dengan shuffle, state sampler-nya juga perlu disimpan.
2. File ditulis secara **atomik**: ke file sementara di folder yang sama, lalu diganti namanya dengan `os.replace`. Kalau proses mati di tengah penulisan, checkpoint lama tetap utuh dan file setengah jadi tidak pernah terbaca sebagai checkpoint.
3. Checkpoint terbaru dicari dengan kunci **numerik**, bukan `sorted()` biasa. Alasannya dibedah setelah eksperimen.
4. Resume mulai dari `step + 1`. Opsi `--resume-offset 0` meniru pola buku yang mengulang dari `step`; opsi ini kita pakai sebagai kontrol.
5. Sinyal hanya mengangkat bendera. Checkpoint ditulis di batas step berikutnya, sehingga tidak ada step yang tersimpan setengah jalan.

Skrip memakai GPU yang sama dengan notebook dan AMP sesuai kartu (`CHILD_AMP` dari gerbang GPU). Opsi algoritma deterministik PyTorch diaktifkan dalam mode peringatan; kalau masih ada kernel GPU yang nondeterministik, perbandingan kurva memakai toleransi, bukan janji identik.

In [11]:
%%writefile "{WORK}/ch10_child.py"
"""Training mini yang bisa dihentikan lalu dilanjutkan (Bab 10, LLM Design Patterns)."""
import argparse
import os
import re
import signal
import sys
import time

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # syarat cuBLAS deterministik, sebelum CUDA dipakai

import torch
from transformers import GPT2Config, GPT2LMHeadModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

parser = argparse.ArgumentParser()
parser.add_argument("--data", required=True)
parser.add_argument("--ckdir", required=True)
parser.add_argument("--log", required=True)
parser.add_argument("--device", required=True, choices=["cuda", "cpu"])
parser.add_argument("--amp", default="off", choices=["off", "fp16", "bf16"])
parser.add_argument("--steps", type=int, default=60)
parser.add_argument("--save-every", type=int, default=10)
parser.add_argument("--resume-offset", type=int, default=1)
parser.add_argument("--throttle", type=float, default=0.0)
args = parser.parse_args()

torch.use_deterministic_algorithms(True, warn_only=True)
device = torch.device(args.device)
if device.type == "cuda":
    if not torch.cuda.is_available():
        sys.exit("CUDA diminta tetapi tidak tersedia di proses anak")
    probe = torch.ones(1024, device=device) * 2
    torch.cuda.synchronize()
    assert float(probe.sum()) == 2048.0
amp_dtype = {"off": None, "fp16": torch.float16, "bf16": torch.bfloat16}[args.amp]

torch.manual_seed(42)
data = torch.load(args.data, weights_only=True)
blocks = data["blocks"]
order = torch.randperm(len(blocks), generator=torch.Generator().manual_seed(1234))
BATCH = 8


def batch_for(step):
    # kursor data: batch ditentukan sepenuhnya oleh nomor step
    start = (step * BATCH) % len(blocks)
    return blocks[order[start:start + BATCH]].to(device)


cfg = GPT2Config(vocab_size=data["vocab"], n_positions=blocks.shape[1], n_embd=64, n_layer=2, n_head=2,
                 resid_pdrop=0.1, embd_pdrop=0.1, attn_pdrop=0.1, bos_token_id=0, eos_token_id=0)
cfg._attn_implementation = "eager"                  # atensi matmul biasa: backward tanpa kernel fused
model = GPT2LMHeadModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min(1.0, (s + 1) / 10))
scaler = torch.amp.GradScaler(device.type, enabled=(args.amp == "fp16"))

os.makedirs(args.ckdir, exist_ok=True)
CK_NAME = re.compile(r"ck_step_(\d+)\.pt")


def latest_checkpoint():
    found = [(int(m.group(1)), name) for name in os.listdir(args.ckdir) if (m := CK_NAME.fullmatch(name))]
    return os.path.join(args.ckdir, max(found)[1]) if found else None


def save_checkpoint(step):
    state = {
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "rng_cpu": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
    }
    path = os.path.join(args.ckdir, f"ck_step_{step}.pt")
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        torch.save(state, f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)                            # atomik: file lama atau file baru, tidak pernah setengah jadi


start = 0
resume_from = latest_checkpoint()
if resume_from:
    ck = torch.load(resume_from, map_location=device, weights_only=True)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])
    torch.set_rng_state(ck["rng_cpu"].cpu())
    if device.type == "cuda" and ck["rng_cuda"] is not None:
        torch.cuda.set_rng_state(ck["rng_cuda"].cpu(), device)
    start = ck["step"] + args.resume_offset
    print(f"RESUME dari checkpoint step {ck['step']} -> lanjut di step {start}", flush=True)

stop = {"signal": None}


def request_stop(signum, frame):
    stop["signal"] = signum                          # hanya bendera; checkpoint ditulis di batas step


signal.signal(signal.SIGTERM, request_stop)
signal.signal(signal.SIGINT, request_stop)

print(f"MULAI device={next(model.parameters()).device} amp={args.amp} step {start}..{args.steps - 1}", flush=True)
model.train()
last_done = start - 1
with open(args.log, "a") as log:
    for step in range(start, args.steps):
        if stop["signal"] is not None:
            if last_done >= 0:
                save_checkpoint(last_done)
            print(f"SINYAL {stop['signal']} diterima: checkpoint step {last_done}, keluar rapi", flush=True)
            sys.exit(0)
        x = batch_for(step)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            loss = model(input_ids=x, labels=x).loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"loss non-finite di step {step}")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        log.write(f"{step},{loss.item():.6f}\n")
        log.flush()
        last_done = step
        if step % args.save_every == 0:
            save_checkpoint(step)
        if args.throttle:
            time.sleep(args.throttle)
print("SELESAI", flush=True)

Writing /content/ch10_workspace/ch10_child.py


Sel berikut menyiapkan alat untuk menjalankan skrip itu. `run_child` membuat proses Python baru dan, kalau diminta, mengirim sinyal **hanya ke PID proses tersebut** begitu log-nya mencapai jumlah step tertentu. Setiap proses punya batas waktu; kalau terlewati, proses itu sendiri dihentikan lalu ditunggu sampai benar-benar selesai, sehingga tidak ada proses yatim. Kode keluar juga diperiksa: run normal dan `SIGTERM` harus keluar dengan 0, `SIGKILL` dengan −9.

`throttle` memberi jeda kecil di akhir setiap step agar sinyal sempat datang di tengah training. Jeda ini tidak mengubah perhitungan. Run pertama adalah referensi: training penuh tanpa gangguan.

In [12]:
CHILD = WORK / "ch10_child.py"
DATA = WORK / "corpus_child.pt"
STEPS = range(CFG["child_steps"])
TOL = 1e-3                                           # toleransi |selisih loss|; log mencatat 6 desimal


def count_lines(path):
    if not path.exists():
        return 0
    with open(path) as f:
        return sum(1 for _ in f)


def run_child(tag, ckdir, log, kill=None, kill_after=None, resume_offset=1, timeout=300):
    out_path = WORK / f"{tag}.out"
    cmd = [sys.executable, str(CHILD), "--data", str(DATA), "--ckdir", str(ckdir), "--log", str(log),
           "--device", DEVICE.type, "--amp", CHILD_AMP, "--steps", str(CFG["child_steps"]),
           "--save-every", str(CFG["save_every"]), "--resume-offset", str(resume_offset),
           "--throttle", str(CFG["throttle_s"])]
    env = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONUNBUFFERED="1")
    lines_before = count_lines(log)
    deadline = time.monotonic() + timeout
    with open(out_path, "w") as out:
        proc = subprocess.Popen(cmd, stdout=out, stderr=subprocess.STDOUT, env=env)
        try:
            if kill is not None:
                while count_lines(log) - lines_before < kill_after:
                    if proc.poll() is not None:
                        raise RuntimeError(f"{tag}: proses berakhir sebelum sinyal dikirim (exit {proc.returncode}); lihat {out_path}")
                    if time.monotonic() > deadline:
                        raise TimeoutError(f"{tag}: log tidak mencapai {kill_after} step dalam {timeout} detik")
                    time.sleep(0.01)
                proc.send_signal(signal.SIGTERM if kill == "term" else signal.SIGKILL)   # hanya PID milik proc
            proc.wait(timeout=max(1.0, deadline - time.monotonic()))
        finally:
            if proc.poll() is None:                  # batas waktu terlewati: hentikan proses ini saja, lalu reap
                proc.kill()
                proc.wait()
    expected = {None: 0, "term": 0, "kill": -signal.SIGKILL}[kill]
    text = out_path.read_text()
    if proc.returncode != expected:
        raise RuntimeError(f"{tag}: exit {proc.returncode}, seharusnya {expected}\n{text[-2000:]}")
    return proc.returncode, [l for l in text.splitlines() if l.startswith(("MULAI", "RESUME", "SINYAL", "SELESAI"))]


def read_log(path):
    losses = {}
    for line in path.read_text().splitlines():
        step, value = line.split(",")
        losses[int(step)] = float(value)             # step yang dilatih ulang menimpa nilai lama
    return losses


def max_gap(run, ref, steps):
    return max(abs(run[s] - ref[s]) for s in steps)


def fresh(tag):
    ckdir, log = WORK / f"ck_{tag}", WORK / f"log_{tag}.csv"
    shutil.rmtree(ckdir, ignore_errors=True)
    log.unlink(missing_ok=True)
    return ckdir, log


t0 = time.perf_counter()
ck_ref, log_ref = fresh("ref")
_, lines = run_child("ref", ck_ref, log_ref)
ref = read_log(log_ref)
print(lines[0])
print(f"referensi: {len(ref)} step | loss step 0 = {ref[0]:.4f} -> step {max(ref)} = {ref[max(ref)]:.4f} "
      f"| {time.perf_counter() - t0:.1f} detik termasuk start proses")
assert sorted(ref) == list(STEPS)

MULAI device=cuda:0 amp=fp16 step 0..59
referensi: 60 step | loss step 0 = 8.3085 -> step 59 = 5.8872 | 14.9 detik termasuk start proses


Baris `MULAI` adalah bukti dari dalam proses anak: device tempat model berada dan presisi AMP yang dipakai. Loss awal seharusnya dekat ln 4096 ≈ 8,32, yaitu loss model yang menebak acak di antara 4.096 token. Di validasi lokal penyusun (CPU, profil 40 step) loss turun dari 8,31 ke 6,30, tanda model mini benar-benar belajar dari data, bukan hanya berjalan.

### Eksperimen 1: `SIGTERM`

Proses kedua menjalankan konfigurasi yang sama dan menerima `SIGTERM` setelah `kill_after` step tercatat di log. Kalau mekanismenya benar, proses menyimpan checkpoint dari step terakhir yang selesai, keluar dengan kode 0, lalu proses ketiga melanjutkan tepat satu step sesudahnya. Ukuran keberhasilannya: tidak ada step yang hilang atau dilatih ulang, dan selisih loss terhadap referensi tidak melebihi `TOL`.

In [13]:
ck_term, log_term = fresh("term")
_, lines = run_child("term_killed", ck_term, log_term, kill="term", kill_after=CFG["kill_after"])
print("dihentikan:", lines[-1])
_, lines = run_child("term_resume", ck_term, log_term)
print("resume    :", next(l for l in lines if l.startswith("RESUME")))

term = read_log(log_term)
missing = [s for s in STEPS if s not in term]
repeated = count_lines(log_term) - len(term)
gap_term = max_gap(term, ref, STEPS)
print(f"step hilang: {len(missing)} | step dilatih ulang: {repeated} | maks |loss - referensi|: {gap_term:.2e} (TOL {TOL:g})")
assert not missing and repeated == 0 and gap_term <= TOL

dihentikan: SINYAL 15 diterima: checkpoint step 34, keluar rapi
resume    : RESUME dari checkpoint step 34 -> lanjut di step 35
step hilang: 0 | step dilatih ulang: 0 | maks |loss - referensi|: 0.00e+00 (TOL 0.001)


Handler hanya mengangkat bendera. Loop menyelesaikan step yang sedang berjalan, menulis checkpoint di batas step, lalu keluar dengan kode 0. Proses berikutnya membaca checkpoint itu, memulihkan bobot, momen AdamW, scheduler, scaler, RNG, dan kursor data, lalu lanjut satu step sesudahnya. Karena semua keadaan pulih, kurva gabungan seharusnya tidak bisa dibedakan dari referensi. Di validasi lokal penyusun (CPU, fp32), selisih maksimumnya 0 (identik pada enam desimal log). Di GPU, kernel yang nondeterministik bisa memunculkan selisih kecil; yang penting selisih itu tetap di bawah `TOL` dan jauh di bawah selisih kontrol pada eksperimen berikut.

### Eksperimen 2: `SIGKILL`, dengan kontrol off-by-one

`SIGKILL` tidak memberi kesempatan apa pun. Yang tersisa hanya checkpoint periodik terakhir (setiap 10 step), sehingga semua step sesudahnya harus dilatih ulang. Sesaat setelah proses mati, folder checkpoint-nya kita salin. Folder asli dilanjutkan dengan cara yang benar, sedangkan salinannya dilanjutkan dengan pola buku (`--resume-offset 0`: mengulang dari step yang sudah tersimpan). Salinan ini adalah kontrol: kalau `TOL` masuk akal, resume yang benar harus lolos dan resume off-by-one harus gagal.

In [14]:
ck_kill, log_kill = fresh("kill")
code_kill, _ = run_child("kill_killed", ck_kill, log_kill, kill="kill", kill_after=CFG["kill_after"])
last_logged = max(read_log(log_kill))
matches = [re.fullmatch(r"ck_step_(\d+)\.pt", n) for n in os.listdir(ck_kill)]
autosaves = sorted(int(m.group(1)) for m in matches if m)
leftover_tmp = [n for n in os.listdir(ck_kill) if n.endswith(".tmp")]
print(f"exit code {code_kill} | step terakhir di log: {last_logged} | checkpoint selamat: {autosaves} | file .tmp: {leftover_tmp}")

ck_obo, log_obo = fresh("obo")
shutil.copytree(ck_kill, ck_obo)                     # salinan untuk kontrol off-by-one
shutil.copy(log_kill, log_obo)

_, lines = run_child("kill_resume", ck_kill, log_kill)
print("resume benar   :", next(l for l in lines if l.startswith("RESUME")))
_, lines = run_child("obo_resume", ck_obo, log_obo, resume_offset=0)
print("resume ala buku:", next(l for l in lines if l.startswith("RESUME")))

kill_run, obo_run = read_log(log_kill), read_log(log_obo)
after = range(autosaves[-1] + 1, CFG["child_steps"])
gap_kill = max_gap(kill_run, ref, STEPS)
gap_obo = max_gap(obo_run, ref, after)
print(f"step dilatih ulang: {last_logged - autosaves[-1]} (step {autosaves[-1] + 1}..{last_logged})")
print(f"maks |loss - referensi| -> resume benar: {gap_kill:.2e} | resume off-by-one: {gap_obo:.2e} | TOL {TOL:g}")
assert sorted(kill_run) == list(STEPS) and gap_kill <= TOL < gap_obo

exit code -9 | step terakhir di log: 34 | checkpoint selamat: [0, 10, 20, 30] | file .tmp: []
resume benar   : RESUME dari checkpoint step 30 -> lanjut di step 31
resume ala buku: RESUME dari checkpoint step 30 -> lanjut di step 30
step dilatih ulang: 4 (step 31..34)
maks |loss - referensi| -> resume benar: 0.00e+00 | resume off-by-one: 4.61e-02 | TOL 0.001


Exit code −9 berarti proses dihentikan oleh sinyal 9 (`SIGKILL`) tanpa sempat mencetak apa pun. Checkpoint terakhir yang selamat adalah kelipatan 10 sebelum titik mati, jadi beberapa step sesudahnya hangus dan dilatih ulang. Inilah aritmetika frekuensi checkpoint: dengan interval N step, satu kegagalan keras membuang paling banyak N − 1 step kerja. Memilih N berarti menimbang biaya I/O setiap penyimpanan dengan biaya komputasi yang rela diulang.

Step yang dilatih ulang tetap menghasilkan kurva yang menempel pada referensi, karena RNG dan kursor data ikut dipulihkan. Kontrolnya menunjukkan sebaliknya. Mengulang dari step yang sudah tersimpan berarti satu batch dilatih dua kali dan AdamW mengambil langkah ekstra, sehingga lintasannya bergeser permanen. Di validasi lokal penyusun (CPU), resume yang benar berselisih 0 (identik pada enam desimal log) dari referensi, sedangkan kontrol off-by-one berselisih 0,062, sekitar 62 kali `TOL`. Artinya toleransi 1e-3 cukup ketat untuk menangkap kesalahan resume yang nyata, tetapi tetap memberi ruang bagi nondeterminisme kecil di GPU.

### Mencari checkpoint terbaru: `sorted()` versus kunci numerik

`get_latest_checkpoint` di buku mengambil elemen terakhir dari `sorted(os.listdir(...))`. Sel berikut membuat empat file kosong bernama seperti checkpoint epoch 8 sampai 11, lalu membandingkan dua cara mengurutkannya. Ini murni operasi string di CPU.

In [15]:
lex_dir = WORK / "ck_lex"
shutil.rmtree(lex_dir, ignore_errors=True)
lex_dir.mkdir()
for e in (8, 9, 10, 11):
    (lex_dir / f"checkpoint_epoch_{e}_step_0.pt").write_text("x")

names = os.listdir(lex_dir)
by_string = sorted(names)
by_number = sorted(names, key=lambda s: tuple(int(x) for x in re.findall(r"\d+", s)))
print("sorted() biasa :", by_string)
print("terbaru (buku) :", by_string[-1])
print("terbaru (angka):", by_number[-1])
print("kasus step     :", sorted(["ck_step_45.pt", "ck_step_5.pt"])[-1], "dianggap lebih baru daripada ck_step_45.pt")
assert by_string[-1] == "checkpoint_epoch_9_step_0.pt" and by_number[-1] == "checkpoint_epoch_11_step_0.pt"

sorted() biasa : ['checkpoint_epoch_10_step_0.pt', 'checkpoint_epoch_11_step_0.pt', 'checkpoint_epoch_8_step_0.pt', 'checkpoint_epoch_9_step_0.pt']
terbaru (buku) : checkpoint_epoch_9_step_0.pt
terbaru (angka): checkpoint_epoch_11_step_0.pt
kasus step     : ck_step_5.pt dianggap lebih baru daripada ck_step_45.pt


`sorted()` membandingkan string karakter demi karakter. Pada posisi angka epoch, `'9'` lebih besar daripada `'1'`, sehingga `epoch_9` berada di urutan terakhir dan dianggap terbaru, padahal training sudah sampai epoch 11. Tidak ada pesan error; training hanya diam-diam melanjutkan dari dua epoch yang lalu. Bug ini tidak terlihat selama demo masih di epoch dan step satu digit, dan baru muncul saat run serius melewati angka 10. Kunci numerik (ambil angkanya, bandingkan sebagai bilangan) menyelesaikannya, dan skrip anak di atas memakai cara ini. Alternatif lain: beri nomor dengan nol di depan (`step_000045`) atau simpan penunjuk `latest` secara eksplisit.

## Checkpoint pada training terdistribusi

Pada training multi-GPU, setiap GPU dijalankan oleh satu proses yang disebut **rank**. Resep buku: hanya rank 0 yang menulis checkpoint (agar beberapa proses tidak menulis file yang sama), semua rank menunggu di `dist.barrier()`, dan saat resume rank 0 membaca checkpoint lalu menyiarkan (*broadcast*) isinya ke rank lain.

> **Listing buku: inisialisasi khusus NCCL**
>
> ```python
> def init_distributed():
>     dist.init_process_group(backend='nccl')
>     rank = dist.get_rank()
>     torch.cuda.set_device(rank)
>     return rank
> ```
>
> Fungsi ini mengandalkan variabel lingkungan dari peluncur seperti `torchrun` dan satu GPU per rank. Di build PyTorch CPU ia gagal dengan `RuntimeError: Distributed package doesn't have NCCL built in` (tercatat di run historis notebook asal). NCCL juga mengharuskan setiap rank memakai GPU berbeda, jadi demo dua rank tidak bisa memakai NCCL di Colab satu GPU. Notebook ini tidak membuat process group di kernel utama; demo berikut berjalan di subprocess dengan backend `gloo`.

Yang ingin diuji adalah isi broadcast-nya. Di `load_checkpoint_distributed` buku, yang dikirim rank 0 ke rank lain hanya epoch, step, dan loss, tiga skalar. Bobot yang dimuat rank 0 tidak pernah dikirim. Skrip di bawah meluncurkan dua proses lokal (CPU, `gloo`, hanya alamat 127.0.0.1) dan membandingkan tiga cara resume: pola buku, semua rank memuat file yang sama, dan rank 0 memuat lalu menyiarkan setiap tensor. Ini bagian CPU yang disengaja: yang diuji adalah logika komunikasi, bukan kecepatan GPU.

In [16]:
%%writefile "{WORK}/ch10_dist.py"
"""Dua rank lokal (CPU, gloo) untuk menguji isi broadcast saat resume (Bab 10)."""
import os
import sys

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from transformers import GPT2Config, GPT2LMHeadModel
from transformers.utils import logging as hf_logging


def checksum(model):
    return float(sum(p.detach().double().sum() for p in model.parameters()))


def run(rank, world_size, mode, port, ckpt_path):
    hf_logging.set_verbosity_error()
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = port
    dist.init_process_group("gloo", rank=rank, world_size=world_size)
    try:
        torch.manual_seed(100 + rank)                # proses baru pasca-crash: inisialisasi berbeda per rank
        cfg = GPT2Config(vocab_size=512, n_positions=32, n_embd=32, n_layer=1, n_head=2, bos_token_id=0, eos_token_id=0)
        model = torch.nn.parallel.DistributedDataParallel(GPT2LMHeadModel(cfg))
        ck = torch.load(ckpt_path, weights_only=True) if (rank == 0 or mode == "all_load") else None
        if mode == "book":                           # rank 0 memuat, hanya skalar yang disiarkan
            if rank == 0:
                model.load_state_dict(ck["model_state_dict"])
            meta = torch.tensor([ck["epoch"], ck["step"], ck["loss"]] if rank == 0 else [0.0, 0.0, 0.0], dtype=torch.float64)
            dist.broadcast(meta, src=0)
        elif mode == "all_load":                     # setiap rank memuat file yang sama
            model.load_state_dict(ck["model_state_dict"])
        elif mode == "broadcast":                    # rank 0 memuat, lalu menyiarkan setiap tensor
            if rank == 0:
                model.load_state_dict(ck["model_state_dict"])
            for tensor in model.state_dict().values():
                dist.broadcast(tensor, src=0)
        dist.barrier()
        print(f"{mode} rank {rank} {checksum(model):+.6f}", flush=True)
    finally:
        dist.destroy_process_group()


if __name__ == "__main__":
    mode, port, ckpt_path = sys.argv[1], sys.argv[2], sys.argv[3]
    mp.spawn(run, args=(2, mode, port, ckpt_path), nprocs=2, join=True)

Writing /content/ch10_workspace/ch10_dist.py


Checksum di sini adalah jumlah seluruh parameter model dalam presisi ganda. Dua model dengan bobot identik pasti punya checksum sama, jadi angka ini cukup untuk mendeteksi rank yang memegang bobot berbeda. Kedua rank sengaja diinisialisasi dari seed berbeda, seperti proses baru setelah crash. Saat dibungkus `DistributedDataParallel`, bobot awal rank 0 disalin sekali ke rank lain, lalu checkpoint dimuat dengan tiga cara.

Setiap mode berjalan di subprocess dengan sesi proses sendiri dan batas waktu; kalau terlewati, hanya grup proses milik subprocess itu yang dihentikan. Set `RUN_DISTRIBUTED_DEMO = False` untuk melewati demo ini.

In [17]:
RUN_DISTRIBUTED_DEMO = True


def free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def run_dist(mode, ckpt_path, timeout=180):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1")
    cmd = [sys.executable, str(WORK / "ch10_dist.py"), mode, str(free_port()), str(ckpt_path)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=env, start_new_session=True)
    try:
        out, err = proc.communicate(timeout=timeout)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)          # grup proses milik subprocess ini saja
        proc.communicate()
        raise
    if proc.returncode != 0:
        raise RuntimeError(f"demo {mode} gagal (exit {proc.returncode}):\n{err[-2000:]}")
    return out


torch.manual_seed(7)                                 # "hasil training" yang akan di-resume (CPU)
ref_model = GPT2LMHeadModel(GPT2Config(vocab_size=512, n_positions=32, n_embd=32, n_layer=1, n_head=2,
                                       bos_token_id=0, eos_token_id=0))
dist_ckpt = WORK / "dist_ck.pt"
torch.save({"epoch": 5, "step": 500, "loss": 2.5,
            "model_state_dict": {"module." + k: v for k, v in ref_model.state_dict().items()}}, dist_ckpt)
expected = float(sum(p.detach().double().sum() for p in ref_model.parameters()))
print(f"checksum checkpoint: {expected:+.6f}")

if RUN_DISTRIBUTED_DEMO:
    sums = {}
    for mode in ("book", "all_load", "broadcast"):
        for line in run_dist(mode, dist_ckpt).splitlines():
            parts = line.split()
            if len(parts) == 4 and parts[1] == "rank":
                sums[(parts[0], int(parts[2]))] = float(parts[3])
    for (mode, rank), value in sorted(sums.items()):
        verdict = "= checkpoint" if abs(value - expected) < 1e-6 else "BEDA dari checkpoint"
        print(f"{mode:<10} rank {rank}: checksum {value:+.6f}  {verdict}")
    assert abs(sums[("book", 0)] - expected) < 1e-6 < abs(sums[("book", 1)] - expected)
    assert all(abs(sums[(m, r)] - expected) < 1e-6 for m in ("all_load", "broadcast") for r in (0, 1))
else:
    print("Demo terdistribusi dilewati (RUN_DISTRIBUTED_DEMO = False); tidak ada hasil untuk dibaca.")

checksum checkpoint: +93.685427
all_load   rank 0: checksum +93.685427  = checkpoint
all_load   rank 1: checksum +93.685427  = checkpoint
book       rank 0: checksum +93.685427  = checkpoint
book       rank 1: checksum +97.227620  BEDA dari checkpoint
broadcast  rank 0: checksum +93.685427  = checkpoint
broadcast  rank 1: checksum +93.685427  = checkpoint


Pola buku menghasilkan dua checksum berbeda: rank 0 memegang bobot checkpoint, sedangkan rank 1 masih memegang bobot awal. `dist.barrier()` hanya membuat semua proses saling menunggu; ia tidak menyamakan isi model. Masalah ini tidak hilang dengan sendirinya, karena DDP menyinkronkan **gradien** di setiap step, bukan parameter. Dua replika yang mulai dari bobot berbeda menerima update yang sama dan tetap berbeda selamanya, tanpa satu pun error.

Kedua perbaikan menghasilkan checksum yang identik dengan checkpoint di kedua rank. Semua rank memuat file yang sama adalah cara paling sederhana kalau file ada di penyimpanan bersama. Rank 0 memuat lalu menyiarkan setiap tensor cocok kalau hanya rank 0 yang bisa membaca file.

## Versioning checkpoint

Fitur berikutnya memberi nama pada checkpoint (misalnya `v1.0`), mencatat riwayatnya di file JSON, dan mendukung cabang eksperimen. Di buku, `VersionControlledLLMTrainer` mewarisi `DistributedLLMTrainer`, yang mewarisi `EfficientLLMTrainer`.

> **Listing buku: versioning yang ikut membutuhkan process group**
>
> ```python
> class DistributedLLMTrainer(EfficientLLMTrainer):
>     def __init__(self, model, optimizer, checkpoint_dir='checkpoints', ...):
>         super().__init__(model, optimizer, checkpoint_dir)
>         self.rank = dist.get_rank()
> ```
>
> Di satu proses tanpa `init_process_group`, membuat `VersionControlledLLMTrainer` berhenti dengan `ValueError: Default process group has not been initialized`, padahal versioning tidak butuh rank. Setelah itu muncul masalah kedua: `save_checkpoint_efficient` di buku tidak mengembalikan path, sehingga `versions.json` mencatat `"path": null` dan `load_checkpoint_versioned` gagal saat membuka path tersebut. Perbaikannya: rank dianggap 0 kalau process group belum dibuat, dan `EfficientLLMTrainer` di atas sudah memakai `return checkpoint_path`.

Sel berikut memuat bagian `DistributedLLMTrainer` yang dibutuhkan (constructor, simpan oleh rank 0 lalu barrier, pencarian checkpoint terbaru dengan kunci numerik, dan loop training dasar), lalu kelas versioning dari buku. Demonya memakai model mini di GPU: simpan `v1.0`, buat cabang, geser bobot, lalu muat cabang dan periksa bobotnya kembali.

In [18]:
import torch.distributed as dist


class DistributedLLMTrainer(EfficientLLMTrainer):
    def __init__(self, model, optimizer, checkpoint_dir='checkpoints', autosave_interval=15):
        super().__init__(model, optimizer, checkpoint_dir)
        self.autosave_interval = autosave_interval
        initialized = dist.is_available() and dist.is_initialized()
        self.rank = dist.get_rank() if initialized else 0            # perbaikan: tanpa process group = rank 0
        self.world_size = dist.get_world_size() if initialized else 1

    def save_checkpoint_distributed(self, epoch, step, loss):
        if self.rank == 0:
            self.save_checkpoint_efficient(epoch, step, loss)
        if self.world_size > 1:
            dist.barrier()

    def get_latest_checkpoint(self):
        cks = [n for n in os.listdir(self.checkpoint_dir) if n.startswith('checkpoint_')]
        cks.sort(key=lambda s: tuple(int(x) for x in re.findall(r'\d+', s)))   # kunci numerik
        return os.path.join(self.checkpoint_dir, cks[-1]) if cks else None

    def train_distributed(self, epochs, steps_per_epoch, train_fn):
        for epoch in range(epochs):
            self.current_epoch = epoch
            for step in range(steps_per_epoch):
                self.current_step = step
                self.current_loss = train_fn(self.model, epoch, step)
                if step % self.autosave_interval == 0:
                    self.save_checkpoint_distributed(epoch, step, self.current_loss)


class VersionControlledLLMTrainer(DistributedLLMTrainer):
    def __init__(self, model, optimizer, checkpoint_dir='checkpoints', version_file='versions.json'):
        super().__init__(model, optimizer, checkpoint_dir)
        self.version_file = version_file
        self.versions = self.load_versions()

    def load_versions(self):
        if os.path.exists(self.version_file):
            with open(self.version_file, 'r') as f:
                return json.load(f)
        return {}

    def save_versions(self):
        with open(self.version_file, 'w') as f:
            json.dump(self.versions, f, indent=2)

    def save_checkpoint_versioned(self, epoch, step, loss, version_name):
        checkpoint_path = self.save_checkpoint_efficient(epoch, step, loss)
        self.versions[version_name] = {'path': checkpoint_path, 'epoch': epoch, 'step': step, 'loss': loss}
        self.save_versions()
        print(f"Saved version '{version_name}': {checkpoint_path}")

    def load_checkpoint_versioned(self, version_name):
        if version_name not in self.versions:
            raise ValueError(f"Version '{version_name}' not found")
        return self.load_checkpoint_efficient(self.versions[version_name]['path'])

    def create_branch(self, base_version, new_version):
        if base_version not in self.versions:
            raise ValueError(f"Base version '{base_version}' not found")
        base_info = self.versions[base_version]
        new_path = os.path.join(self.checkpoint_dir, f"branch_{new_version}.zip")   # buku: .pt untuk arsip zip
        shutil.copy(base_info['path'], new_path)
        self.versions[new_version] = {**base_info, 'path': new_path, 'branched_from': base_version}
        self.save_versions()
        print(f"Created branch '{new_version}' from '{base_version}'")


vc_dir = WORK / 'ck_vc'
shutil.rmtree(vc_dir, ignore_errors=True)
version_file = WORK / 'versions.json'
version_file.unlink(missing_ok=True)
vtrainer = VersionControlledLLMTrainer(model, optimizer, checkpoint_dir=str(vc_dir), version_file=str(version_file))
vtrainer.save_checkpoint_versioned(epoch=10, step=500, loss=0.1, version_name="v1.0")
vtrainer.create_branch("v1.0", "experimental_branch")

weights_v1 = {k: v.detach().clone() for k, v in model.state_dict().items()}
train_step(model, optimizer, batch(0))               # geser bobot sebelum memuat cabang
epoch, step, loss_v = vtrainer.load_checkpoint_versioned("experimental_branch")
same = all(torch.equal(v, weights_v1[k]) for k, v in model.state_dict().items())
print(f"rank {vtrainer.rank} dari world {vtrainer.world_size} | cabang dimuat: epoch {epoch}, step {step}, loss {loss_v} | bobot = v1.0: {same}")
print(version_file.read_text())
assert same and json.loads(version_file.read_text())["experimental_branch"]["branched_from"] == "v1.0"

Efficient checkpoint saved: /content/ch10_workspace/ck_vc/checkpoint_epoch_10_step_500.zip
Saved version 'v1.0': /content/ch10_workspace/ck_vc/checkpoint_epoch_10_step_500.zip
Created branch 'experimental_branch' from 'v1.0'
rank 0 dari world 1 | cabang dimuat: epoch 10, step 500, loss 0.1 | bobot = v1.0: True
{
  "v1.0": {
    "path": "/content/ch10_workspace/ck_vc/checkpoint_epoch_10_step_500.zip",
    "epoch": 10,
    "step": 500,
    "loss": 0.1
  },
  "experimental_branch": {
    "path": "/content/ch10_workspace/ck_vc/branch_experimental_branch.zip",
    "epoch": 10,
    "step": 500,
    "loss": 0.1,
    "branched_from": "v1.0"
  }
}


`versions.json` kini menyimpan path yang nyata, dan cabang `experimental_branch` mencatat asalnya lewat `branched_from`. Memuat cabang mengembalikan bobot persis seperti saat `v1.0` disimpan. Dua penyesuaian kecil: ekstensi file cabang diluruskan menjadi `.zip` (buku menyalin arsip zip ke nama berakhiran `.pt`), dan `versions.json` ditulis ke folder kerja bab, bukan ke direktori kerja Python.

## Checkpoint dan recovery otomatis

Lapisan terakhir di buku menjalankan dua thread di samping loop training: thread autosave berbasis waktu dan thread *health check* yang bisa memicu recovery.

> **Listing buku: satu nama untuk dua satuan**
>
> ```python
> def autosave_loop():
>     while self.training_active:
>         time.sleep(self.autosave_interval)          # dibaca sebagai detik
> # ... sementara di train_distributed yang diwarisi:
> if step % self.autosave_interval == 0:              # dibaca sebagai jumlah step
> ```
>
> `AutomatedLLMTrainer` memakai `autosave_interval` sebagai detik, sedangkan `train_distributed` yang diwarisinya memakai atribut yang sama sebagai interval step. Di run historis notebook asal dengan `autosave_interval=1`, loop step menyimpan di setiap step (25 file untuk 25 step) sementara thread waktu ikut menyimpan, bahkan ke nama file yang sama. Dengan nilai 15 tabrakan ini tersamar karena 15 step dan 15 detik sama-sama tampak wajar.

Versi di bawah memisahkan dua knob: `autosave_every_steps` untuk loop dan `autosave_seconds` untuk thread. Thread waktu kini hanya mengangkat tanda (`threading.Event`); checkpoint-nya ditulis oleh loop utama di batas step berikutnya dengan nama versi sendiri. Ini pola yang sama dengan penanganan `SIGTERM` di skrip anak, dan membuat thread tidak pernah memotret state di tengah `optimizer.step()`. Perubahan ini keputusan desain notebook, bukan hasil pengukuran bug. `time.sleep(0.06)` di `train_fn` adalah jeda buatan agar thread berbasis waktu sempat bekerja dalam run sependek ini.

In [19]:
class AutomatedLLMTrainer(VersionControlledLLMTrainer):
    def __init__(self, model, optimizer, checkpoint_dir='checkpoints', version_file='versions.json',
                 autosave_every_steps=10, autosave_seconds=15, health_check_interval=60):
        super().__init__(model, optimizer, checkpoint_dir, version_file)
        self.autosave_interval = autosave_every_steps     # satuan STEP, dipakai loop training
        self.autosave_seconds = autosave_seconds          # satuan DETIK, dipakai thread
        self.health_check_interval = health_check_interval
        self.training_active = False
        self.save_requested = threading.Event()
        self.timed_saves = 0

    def start_autosave_thread(self):
        def autosave_loop():
            while self.training_active:
                time.sleep(self.autosave_seconds)
                if self.training_active:
                    self.save_requested.set()             # hanya tanda; loop utama yang menulis
        self.autosave_thread = threading.Thread(target=autosave_loop, daemon=True)
        self.autosave_thread.start()

    def start_health_check_thread(self):
        def health_check_loop():
            while self.training_active:
                time.sleep(self.health_check_interval)
                if self.training_active and not self.check_system_health():
                    print("System health check failed. Initiating recovery...")
                    self.initiate_recovery()
        self.health_check_thread = threading.Thread(target=health_check_loop, daemon=True)
        self.health_check_thread.start()

    def check_system_health(self):
        return True

    def initiate_recovery(self):
        pass

    def save_timed_version(self, epoch, step, loss):
        self.timed_saves += 1
        name = f"autosave_{self.timed_saves}"
        path = self.save_checkpoint_efficient(epoch, step, loss, name=name)
        self.versions[name] = {'path': path, 'epoch': epoch, 'step': step, 'loss': loss}
        self.save_versions()

    def train_distributed(self, epochs, steps_per_epoch, train_fn):
        for epoch in range(epochs):
            self.current_epoch = epoch
            for step in range(steps_per_epoch):
                self.current_step = step
                self.current_loss = train_fn(self.model, epoch, step)
                if step % self.autosave_interval == 0:
                    self.save_checkpoint_distributed(epoch, step, self.current_loss)
                if self.save_requested.is_set():          # tambahan: save berbasis waktu di batas step
                    self.save_requested.clear()
                    self.save_timed_version(epoch, step, self.current_loss)

    def train_with_automation(self, epochs, steps_per_epoch, train_fn):
        self.training_active = True
        self.start_autosave_thread()
        self.start_health_check_thread()
        try:
            self.train_distributed(epochs, steps_per_epoch, train_fn)
        finally:
            self.training_active = False
            self.autosave_thread.join()
            self.health_check_thread.join()


def train_fn(m, epoch, step):
    loss = train_step(m, optimizer, batch(step))
    time.sleep(0.06)
    return loss


auto_dir = WORK / 'ck_auto'
shutil.rmtree(auto_dir, ignore_errors=True)
auto_versions = WORK / 'versions_auto.json'
auto_versions.unlink(missing_ok=True)
auto = AutomatedLLMTrainer(model, optimizer, checkpoint_dir=str(auto_dir), version_file=str(auto_versions),
                           autosave_every_steps=10, autosave_seconds=0.5, health_check_interval=1)
t0 = time.perf_counter()
auto.train_with_automation(epochs=1, steps_per_epoch=25, train_fn=train_fn)
elapsed = time.perf_counter() - t0

step_files = sorted((n for n in os.listdir(auto_dir) if n.startswith('checkpoint_')), key=lambda s: int(re.findall(r'\d+', s)[-1]))
thread_files = sorted(n for n in os.listdir(auto_dir) if n.startswith('autosave_'))
print(f"durasi training: {elapsed:.1f} detik di {next(model.parameters()).device}")
print(f"save dari loop (tiap 10 step) : {step_files}")
print(f"save dari thread (tiap 0,5 s) : {len(thread_files)} file -> {thread_files}")
assert step_files == ['checkpoint_epoch_0_step_0.zip', 'checkpoint_epoch_0_step_10.zip', 'checkpoint_epoch_0_step_20.zip']
assert len(thread_files) == auto.timed_saves >= 1

Efficient checkpoint saved: /content/ch10_workspace/ck_auto/checkpoint_epoch_0_step_0.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/autosave_1.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/autosave_2.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/checkpoint_epoch_0_step_10.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/autosave_3.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/autosave_4.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/autosave_5.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/checkpoint_epoch_0_step_20.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/autosave_6.zip
Efficient checkpoint saved: /content/ch10_workspace/ck_auto/autosave_7.zip
durasi training: 4.0 detik di cuda:0
save dari loop (tiap 10 step) : ['checkpoint_epoch_0_step_0.zip', 'checkpoint_epoch_0_step_10.zip', 'checkpoint_epoch_0_step_20.zip']
save dari thread (tiap 0,5 s) : 

Dengan dua knob terpisah, jumlah save bisa ditebak dari konfigurasinya. Loop menghasilkan tepat tiga file untuk 25 step dengan interval 10 (step 0, 10, dan 20). Tanda dari thread muncul kira-kira sekali per 0,5 detik, jadi jumlah save berbasis waktu mendekati durasi training dibagi 0,5 dan bisa berbeda antar mesin. Yang penting, setiap save berbasis waktu punya nama sendiri, tercatat di file versinya, dan ditulis di batas step yang bersih. `check_system_health` dan `initiate_recovery` masih titik sambung kosong seperti di buku; di sistem nyata isinya misalnya pemeriksaan memori GPU dan disk, lalu memuat checkpoint terakhir.

## Ringkasan

- Checkpoint yang bisa melanjutkan training secara kontinu perlu lebih dari bobot: optimizer, scheduler, scaler fp16, RNG, dan kursor data. Uji yang meyakinkan adalah membandingkan kurva loss hasil resume dengan run tanpa gangguan, memakai toleransi yang terbukti cukup ketat lewat kontrol off-by-one.
- `SIGTERM` bisa ditangani tanpa kehilangan step: tangkap sinyal, selesaikan step, simpan, keluar. `SIGKILL` tidak bisa ditangkap; kerugiannya paling banyak satu interval checkpoint, jadi interval itu adalah keputusan biaya.
- Tulis checkpoint secara atomik, cari yang terbaru dengan kunci numerik, dan lanjutkan dari `step + 1`.
- Kompresi ZIP hampir tidak mengecilkan bobot fp32. Ukuran dikendalikan oleh presisi dan deduplikasi, dan `half()` per tensor bisa memutus weight tying sehingga embedding tersimpan dua kali.
- Pada training terdistribusi, menyiarkan skalar tidak menyamakan bobot. Semua rank harus memuat checkpoint yang sama atau menerima setiap tensor dari rank 0.
- Beberapa bug listing buku bersifat diam: `return` yang hilang baru meledak di checkpoint keenam, `sorted()` baru salah ketika angka mencapai dua digit, dan `autosave_interval` dibaca sebagai detik sekaligus step.

Bab berikutnya di buku membahas fine-tuning.